# RSNA Knee MRI — Multimodal Abnormality Detection: EDA

**Goal:** for each *study*, predict 12 binary probabilities. Metric = **macro-averaged AUC-ROC** over the 12 labels.

This notebook answers, in order:

1. What is in each file, and how do they join together?
2. **Labels** — how many studies are actually labeled? prevalence, imbalance, co-occurrence.
3. **Reports** — languages, length, and *can we recover labels from text?* (this is the crux of the competition)
4. **Series metadata** — protocol heterogeneity, which planes/contrasts exist per study.
5. **DICOM pixel level** — transfer syntaxes, geometry, scanners, intensity ranges, decode cost.
6. **Train vs test** — domain shift checks.
7. **Modelling implications** — validation design, what the metric rewards.

Every section ends with a short *Takeaway* cell so the analysis is interpretable, not just plotted.

In [ ]:
import os, re, glob, json, math, random, warnings, time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 300)

mpl.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.grid": True,
    "grid.alpha": .25, "axes.spines.top": False, "axes.spines.right": False,
})

try:
    import seaborn as sns
    sns.set_palette("deep")
    HAS_SNS = True
except ImportError:
    HAS_SNS = False

SEED = 42
random.seed(SEED); np.random.seed(SEED)
print("pandas", pd.__version__, "| numpy", np.__version__, "| seaborn", HAS_SNS)

## 1. Locate the competition directory


In [ ]:
DATA = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
print("DATA =", DATA)
for p in sorted(DATA.glob("*")):
    size = f"{p.stat().st_size/1e6:8.2f} MB" if p.is_file() else "     <dir>"
    print(f"  {size}  {p.name}")

In [ ]:
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
          "Medial OA", "Lateral OA", "PF OA", "Effusion",
          "Synovitis", "Baker's", "Contusion", "Fracture"]

GROUPS = {                      # a clinically meaningful grouping used throughout
    "Ligament":  ["ACL", "MCL"],
    "Meniscus":  ["Medial Meniscus", "Lateral Meniscus"],
    "OA":        ["Medial OA", "Lateral OA", "PF OA"],
    "Inflam":    ["Effusion", "Synovitis", "Baker's"],
    "Trauma":    ["Contusion", "Fracture"],
}
LABEL2GROUP = {l: g for g, ls in GROUPS.items() for l in ls}

train        = pd.read_csv(DATA / "train.csv")
train_series = pd.read_csv(DATA / "train_series.csv")
test         = pd.read_csv(DATA / "test.csv")
test_series  = pd.read_csv(DATA / "test_series.csv")
sample_sub   = pd.read_csv(DATA / "sample_submission.csv")

for name, df in [("train", train), ("train_series", train_series),
                 ("test", test), ("test_series", test_series), ("sample_submission", sample_sub)]:
    print(f"{name:<20} {df.shape}")

In [ ]:
train.head(3)

In [ ]:
# dtypes + missingness for train.csv
info = pd.DataFrame({
    "dtype":     train.dtypes.astype(str),
    "n_missing": train.isna().sum(),
    "pct_missing": (train.isna().mean() * 100).round(2),
    "n_unique":  train.nunique(dropna=True),
})
info

### The single most important structural fact

The description says *"Only a small subset of training studies carry per-condition labels"*. That shows up as **NaN** in the 12 label columns. Everything about strategy follows from this ratio, so quantify it first.

In [ ]:
lab_present = train[LABELS].notna().all(axis=1)
lab_partial  = train[LABELS].notna().any(axis=1) & ~lab_present

n_all, n_lab, n_part = len(train), lab_present.sum(), lab_partial.sum()
print(f"studies total            : {n_all:,}")
print(f"fully labeled            : {n_lab:,}  ({n_lab/n_all:.1%})")
print(f"partially labeled        : {n_part:,}")
print(f"unlabeled (report only)  : {n_all - n_lab - n_part:,}  ({(n_all-n_lab-n_part)/n_all:.1%})")
print(f"has a report             : {train['Report'].notna().sum():,}  ({train['Report'].notna().mean():.1%})")

labeled   = train[lab_present].copy()
unlabeled = train[~lab_present].copy()

---
## 2. Labels: prevalence, imbalance, co-occurrence

In [ ]:
prev = (labeled[LABELS].mean().rename("prevalence").to_frame()
        .assign(n_pos=labeled[LABELS].sum().astype(int),
                n_neg=(len(labeled) - labeled[LABELS].sum()).astype(int),
                group=[LABEL2GROUP[l] for l in LABELS])
        .sort_values("prevalence", ascending=False))
prev["prevalence"] = prev["prevalence"].round(4)
prev

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
p = prev.sort_values("prevalence")
colors = {"Ligament": "#4C72B0", "Meniscus": "#DD8452", "OA": "#55A868",
          "Inflam": "#C44E52", "Trauma": "#8172B3"}
ax.barh(p.index, p["prevalence"], color=[colors[g] for g in p["group"]])
for y, (v, n) in enumerate(zip(p["prevalence"], p["n_pos"])):
    ax.text(v + .005, y, f"{v:.1%} (n={n})", va="center", fontsize=8)
ax.set_xlim(0, p["prevalence"].max() * 1.35)
ax.set_xlabel("positive rate among fully-labeled studies")
ax.set_title("Label prevalence — colour = clinical group")
handles = [plt.Rectangle((0,0),1,1, color=c) for c in colors.values()]
ax.legend(handles, colors.keys(), fontsize=8, loc="lower right", frameon=False)
plt.tight_layout(); plt.show()

**How to read this:** macro-AUC weights every label equally, so a label with only a few dozen positives contributes 1/12 of the score *and* carries most of the variance. Any label under a few hundred positives is where the leaderboard will be won or lost — and also where your local CV will be noisiest.

In [ ]:
# how many findings per study? (comorbidity is the norm in knee MRI)
npos = labeled[LABELS].sum(axis=1)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].hist(npos, bins=np.arange(-.5, npos.max() + 1.5), color="#4C72B0", edgecolor="w")
ax[0].set_xlabel("# positive labels in a study"); ax[0].set_ylabel("studies")
ax[0].set_title(f"Findings per study (mean {npos.mean():.2f})")

grp = pd.DataFrame({g: labeled[ls].max(axis=1) for g, ls in GROUPS.items()})
ax[1].bar(grp.columns, grp.mean(), color="#55A868")
ax[1].set_ylabel("share of studies with >=1 finding"); ax[1].set_title("Prevalence by clinical group")
ax[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

print("completely normal studies (all 12 = 0):", f"{(npos == 0).mean():.1%}")

In [ ]:
# co-occurrence: P(col | row) — asymmetric and far more informative than raw correlation
M = labeled[LABELS].values.astype(float)
joint = M.T @ M
cond = joint / np.clip(M.sum(0)[:, None], 1, None)   # rows = conditioning label
cond_df = pd.DataFrame(cond, index=LABELS, columns=LABELS)

fig, ax = plt.subplots(1, 2, figsize=(13.5, 5.2))
im = ax[0].imshow(cond_df.values, cmap="viridis", vmin=0, vmax=1)
ax[0].set_xticks(range(12)); ax[0].set_xticklabels(LABELS, rotation=90, fontsize=7)
ax[0].set_yticks(range(12)); ax[0].set_yticklabels(LABELS, fontsize=7)
ax[0].set_title("P(column | row)"); ax[0].grid(False)
for i in range(12):
    for j in range(12):
        v = cond_df.values[i, j]
        ax[0].text(j, i, f"{v:.2f}"[1:], ha="center", va="center", fontsize=6,
                   color="w" if v < .6 else "k")
plt.colorbar(im, ax=ax[0], fraction=.046)

# phi (= Pearson on binaries) for symmetric association strength
phi = labeled[LABELS].corr()
im2 = ax[1].imshow(phi.values, cmap="RdBu_r", vmin=-.6, vmax=.6)
ax[1].set_xticks(range(12)); ax[1].set_xticklabels(LABELS, rotation=90, fontsize=7)
ax[1].set_yticks(range(12)); ax[1].set_yticklabels(LABELS, fontsize=7)
ax[1].set_title("phi correlation"); ax[1].grid(False)
plt.colorbar(im2, ax=ax[1], fraction=.046)
plt.tight_layout(); plt.show()

In [ ]:
# strongest pairs, ranked
pairs = (phi.where(np.triu(np.ones(phi.shape), 1).astype(bool))
           .stack().sort_values(ascending=False))
print("STRONGEST POSITIVE ASSOCIATIONS"); print(pairs.head(8).round(3))
print("\nSTRONGEST NEGATIVE ASSOCIATIONS"); print(pairs.tail(5).round(3))

**Takeaway (labels).** Expect the OA compartments to travel together, effusion to co-occur with almost everything acute, and ACL to pull contusion along with it (the classic pivot-shift bone-bruise pattern). Two practical consequences:

* Train **one multi-label head**, not 12 independent models — the shared representation gets label correlation for free.
* Because labels are correlated, a *stacking* layer on the 12 raw logits (or simply averaging within a group) often gains a little AUC on the rare labels.

---
## 3. Reports: the second modality and the label factory

In [ ]:
rep = train["Report"].fillna("")
train["rep_chars"] = rep.str.len()
train["rep_words"] = rep.str.split().str.len()

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].hist(train.loc[train.rep_chars > 0, "rep_chars"], bins=80, color="#4C72B0")
ax[0].set_xlabel("characters"); ax[0].set_title("Report length"); ax[0].set_yscale("log")
ax[1].hist(train.loc[train.rep_words > 0, "rep_words"], bins=80, color="#DD8452")
ax[1].set_xlabel("words"); ax[1].set_title("Report word count"); ax[1].set_yscale("log")
plt.tight_layout(); plt.show()

print(train[["rep_chars", "rep_words"]].describe().round(1))
print("\nempty reports:", (train.rep_chars == 0).sum())

In [ ]:
# --- crude but effective language fingerprinting (no external deps) ---
SCRIPT_RANGES = {
    "Cyrillic": (0x0400, 0x04FF), "Greek": (0x0370, 0x03FF), "Arabic": (0x0600, 0x06FF),
    "Hebrew": (0x0590, 0x05FF), "CJK": (0x4E00, 0x9FFF), "Hangul": (0xAC00, 0xD7AF),
    "Kana": (0x3040, 0x30FF), "Thai": (0x0E00, 0x0E7F), "Devanagari": (0x0900, 0x097F),
}
STOPWORDS = {   # high-signal function words / radiology terms per language
    "English":    ["the ", " and ", " with ", " no ", "knee", "tear", "normal"],
    "Spanish":    [" de ", " del ", " con ", " sin ", "rodilla", "menisco", "rotura"],
    "Portuguese": [" de ", " com ", " sem ", "joelho", "menisco", "sinais"],
    "French":     [" de ", " du ", " avec ", " sans ", "genou", "menisque", "rupture"],
    "German":     [" der ", " und ", " mit ", " ohne ", "kniegelenk", "riss", "kein"],
    "Italian":    [" di ", " con ", " senza ", "ginocchio", "menisco", "lesione"],
    "Dutch":      [" van ", " met ", " geen ", "knie", "meniscus"],
    "Turkish":    [" ve ", "diz ", "menisk", "yirtik"],
}
def script_of(t):
    counts = Counter()
    for ch in t[:1500]:
        o = ord(ch)
        for name, (lo, hi) in SCRIPT_RANGES.items():
            if lo <= o <= hi:
                counts[name] += 1
    return counts.most_common(1)[0][0] if counts and counts.most_common(1)[0][1] > 8 else "Latin"

def latin_lang(t):
    t = " " + t.lower() + " "
    scores = {L: sum(t.count(w) for w in ws) for L, ws in STOPWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] >= 2 else "Latin-unknown"

train["script"] = rep.map(script_of)
train["lang_guess"] = np.where(train.script == "Latin", rep.map(latin_lang), train.script)

fig, ax = plt.subplots(figsize=(7, 3.4))
vc = train.lang_guess.value_counts()
ax.bar(vc.index, vc.values, color="#55A868")
ax.set_yscale("log"); ax.set_ylabel("studies (log)"); ax.tick_params(axis="x", rotation=45)
ax.set_title("Heuristic language / script of report")
plt.tight_layout(); plt.show()
display(vc.to_frame("n").assign(pct=(vc / vc.sum() * 100).round(1)))

In [ ]:
# eyeball a few reports across languages
for lang in train.lang_guess.value_counts().head(4).index:
    s = train.loc[train.lang_guess == lang, "Report"].dropna()
    if not len(s): continue
    print("=" * 90); print(f"### {lang}"); print("=" * 90)
    print(s.iloc[0][:900].strip(), "\n")

### Can we recover labels from the report text?

This is the highest-leverage question in the whole EDA. If a keyword rule reproduces the human labels well on the labeled subset, you can **pseudo-label the entire unlabeled training set** and multiply your effective training data.

The rule below is deliberately simple — English + Spanish/Portuguese/French stems, with a **negation window** — and we score it against ground truth so you know exactly how much to trust it.

In [ ]:
# --- multilingual keyword patterns (extend these; this is a starting point) ---
PAT = {
 "ACL":              r"\b(acl|lca|anterior cruciate|cruzado anterior|croise anterieur|kreuzband)\b",
 "MCL":              r"\b(mcl|lcm|medial collateral|colateral medial|collateral medial|innenband)\b",
 "Medial Meniscus":  r"(medial meniscus|menisco (interno|medial)|menisque (interne|medial)|innenmeniskus)",
 "Lateral Meniscus": r"(lateral meniscus|menisco (externo|lateral)|menisque (externe|lateral)|aussenmeniskus)",
 "Medial OA":        r"(medial (compartment )?(osteoarthr|arthros|chondral loss|degenerative)|(gonartrosis|artrosis).{0,30}medial)",
 "Lateral OA":       r"(lateral (compartment )?(osteoarthr|arthros|chondral loss|degenerative)|(gonartrosis|artrosis).{0,30}lateral)",
 "PF OA":            r"(patellofemoral (osteoarthr|arthros|chondro|degenerativ)|femoropatelar|chondromalacia)",
 "Effusion":         r"(effusion|derrame|epanchement|erguss|joint fluid|liquido articular)",
 "Synovitis":        r"(synovitis|sinovitis|synovite|synovial (thicken|proliferat))",
 "Baker's":          r"(baker|popliteal cyst|quiste de baker|kyste poplite|poplitealzyste)",
 "Contusion":        r"(contusion|bone (bruise|marrow edema|marrow oedema)|edema (oseo|de la medula)|knochenmarkodem)",
 "Fracture":         r"(fracture|fractura|fratura|fraktur|avulsion)",
}
NEG = r"(no |not |without |absence of |negative for |sin |sem |pas de |kein |intact|normal|unremarkable|preserv)"
TEAR = r"(tear|torn|rupture|rotura|ruptura|desgarro|lesion|riss|signal.{0,15}grade (2|3|iii|ii))"

def negated(text, span_start, window=45):
    return re.search(NEG, text[max(0, span_start - window):span_start]) is not None

def rule_predict(text):
    t = re.sub(r"\s+", " ", str(text).lower())
    out = {}
    for lab, pat in PAT.items():
        hit = 0
        for m in re.finditer(pat, t):
            if negated(t, m.start()):
                continue
            # ligaments & menisci need an injury word nearby, otherwise it is just anatomy description
            if lab in ("ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"):
                ctx = t[m.start(): m.end() + 90]
                if not re.search(TEAR, ctx):
                    continue
            hit = 1; break
        out[lab] = hit
    return out

rule_df = pd.DataFrame(list(labeled["Report"].fillna("").map(rule_predict)), index=labeled.index)
print("rule-based predictions computed for", len(rule_df), "labeled studies")

In [ ]:
def prf(y, p):
    tp = int(((y == 1) & (p == 1)).sum()); fp = int(((y == 0) & (p == 1)).sum())
    fn = int(((y == 1) & (p == 0)).sum())
    prec = tp / (tp + fp) if tp + fp else np.nan
    rec  = tp / (tp + fn) if tp + fn else np.nan
    f1   = 2 * prec * rec / (prec + rec) if prec and rec else np.nan
    return prec, rec, f1, tp, fp, fn

rows = []
for lab in LABELS:
    y = labeled[lab].astype(int).values
    p = rule_df[lab].values
    prec, rec, f1, tp, fp, fn = prf(y, p)
    rows.append(dict(label=lab, prevalence=y.mean(), precision=prec, recall=rec,
                     f1=f1, TP=tp, FP=fp, FN=fn))
rule_eval = pd.DataFrame(rows).set_index("label").round(3)
display(rule_eval.sort_values("f1", ascending=False))
print(f"macro F1 = {rule_eval.f1.mean():.3f} | macro precision = {rule_eval.precision.mean():.3f} "
      f"| macro recall = {rule_eval.recall.mean():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
r = rule_eval.sort_values("f1")
ax.barh(r.index, r["precision"], height=.38, label="precision", color="#4C72B0")
ax.barh([i + .4 for i in range(len(r))], r["recall"], height=.38, label="recall", color="#DD8452")
ax.set_yticks([i + .2 for i in range(len(r))]); ax.set_yticklabels(r.index, fontsize=8)
ax.set_xlim(0, 1); ax.legend(frameon=False); ax.set_title("Keyword rule vs. ground-truth labels")
plt.tight_layout(); plt.show()

In [ ]:
# inspect the failures — this is how you improve the patterns
def show_errors(lab, kind="FN", k=3):
    y = labeled[lab].astype(int).values; p = rule_df[lab].values
    mask = (y == 1) & (p == 0) if kind == "FN" else (y == 0) & (p == 1)
    idx = np.where(mask)[0][:k]
    print("#" * 100); print(f"### {lab} — {kind} examples ({mask.sum()} total)"); print("#" * 100)
    for i in idx:
        txt = re.sub(r"\s+", " ", str(labeled.iloc[i]["Report"]))
        print("-" * 90); print(txt[:700])

worst = rule_eval.f1.idxmin()
show_errors(worst, "FN", 3)
show_errors(worst, "FP", 2)

**Takeaway (text).** Read the F1 table as a *ceiling estimate for pseudo-labels*. Labels where precision is high but recall mediocre are safe to use as **positive-only** pseudo-labels; labels where precision is poor should be pseudo-labeled with soft targets or excluded.

A stronger production path, in increasing order of effort:

1. Extend these regexes per language using the FN/FP inspector above (cheap, gets you a long way).
2. Machine-translate everything to English, then run one English rule set.
3. Fine-tune a multilingual encoder (XLM-R / mDeBERTa) on the *labeled* subset using the report as input, then run it over the unlabeled reports to get **soft** pseudo-labels for the image model.

Critically: **reports exist only for training.** At test time you get pixels alone. Text is a teacher, never an input to the final model.

In [ ]:
# quick corpus view: most frequent tokens (after stripping numerics)
tok = Counter()
for t in rep.sample(min(4000, (rep.str.len() > 0).sum()), random_state=SEED):
    tok.update(w for w in re.findall(r"[a-zA-Zaaaaeeeiiooouuunc]{4,}", str(t).lower()))
pd.Series(dict(tok.most_common(40))).to_frame("count")

---
## 4. Series metadata: what protocol does each study actually contain?

In [ ]:
display(train_series.head())
print(train_series.dtypes)
print("\nseries rows:", len(train_series), "| unique studies:", train_series.StudyInstanceUID.nunique())
print("studies in train.csv missing from train_series.csv:",
      len(set(train.StudyInstanceUID) - set(train_series.StudyInstanceUID)))

In [ ]:
spc = train_series.groupby("StudyInstanceUID").size()
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].hist(spc, bins=np.arange(.5, spc.max() + 1.5), color="#4C72B0")
ax[0].set_title(f"Series per study (median {spc.median():.0f})"); ax[0].set_xlabel("# series")

vc = train_series.Anatomical_Plane.value_counts(dropna=False)
ax[1].bar(vc.index.astype(str), vc.values, color="#DD8452"); ax[1].set_title("Plane")

combo = (train_series.assign(k=train_series.Fluid_Sensitive.astype(str) + "/" +
                              train_series.Fat_Suppression.astype(str))
         .k.value_counts())
ax[2].bar(combo.index, combo.values, color="#55A868")
ax[2].set_title("Fluid_Sensitive / Fat_Suppression"); ax[2].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

In [ ]:
# full protocol cross-tab: plane x contrast type
ct = pd.crosstab(train_series.Anatomical_Plane,
                 train_series.Fluid_Sensitive.astype(str) + "|" + train_series.Fat_Suppression.astype(str))
display(ct)
print("\nrow-normalised:"); display((ct.div(ct.sum(1), axis=0) * 100).round(1))

In [ ]:
# per-study protocol fingerprint: which (plane, fluid, fatsat) combos are present?
train_series["combo"] = (train_series.Anatomical_Plane.astype(str) + "_F" +
                         train_series.Fluid_Sensitive.astype(str) + "S" +
                         train_series.Fat_Suppression.astype(str))
pivot = (train_series.assign(v=1)
         .pivot_table(index="StudyInstanceUID", columns="combo", values="v",
                      aggfunc="max", fill_value=0))
cov = pivot.mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(cov.index[::-1], cov.values[::-1], color="#8172B3")
ax.set_xlabel("fraction of studies containing this sequence type")
ax.set_title("Sequence-type coverage across studies")
plt.tight_layout(); plt.show()
display(cov.to_frame("coverage").round(3))

In [ ]:
# how many studies are missing each plane entirely?
planes = (train_series.pivot_table(index="StudyInstanceUID", columns="Anatomical_Plane",
                                   values="SeriesInstanceUID", aggfunc="count", fill_value=0) > 0)
print("studies missing each plane:")
print((~planes).mean().round(3).to_frame("fraction_missing"))
print("\nmost common plane-sets:")
print(planes.apply(lambda r: "+".join(sorted(planes.columns[r.values])), axis=1)
            .value_counts().head(8))

**Takeaway (protocol).** You cannot assume a fixed input tensor. Some studies lack an axial series, some lack fat-suppressed sequences, some have duplicates. Two robust designs:

* **Per-sequence-type slots with dropout.** Define canonical slots (Sag-fluid-fatsat, Cor-fluid-fatsat, Ax-fluid-fatsat, Sag-T1…), fill what exists, zero-fill and mask what does not, and randomly drop slots during training so the model never depends on one.
* **Series-agnostic MIL.** Encode every series independently, then attention-pool over all series in the study. Handles variable counts natively and is what usually wins these study-level tasks.

Also note which findings live in which plane — coronal fluid-sensitive for MCL and meniscal roots, sagittal for ACL, axial for patellofemoral cartilage. If you must subsample, subsample with anatomy in mind, not at random.

---
## 5. DICOM level: geometry, scanners, encoding, intensities

In [ ]:
try:
    import pydicom
    from pydicom.uid import UID
    HAS_DCM = True
except ImportError:
    HAS_DCM = False
    print("pydicom unavailable — skip section 5")

SERIES_DIR = DATA / "train_series"
print("series dir exists:", SERIES_DIR.exists())

In [ ]:
N_STUDIES_SCAN = 60        # raise for a more reliable picture; each study is a few file reads

def list_slices(study, series):
    return sorted((SERIES_DIR / study / series).glob("*.dcm"))

if HAS_DCM and SERIES_DIR.exists():
    study_ids = [s for s in train_series.StudyInstanceUID.unique() if (SERIES_DIR / s).exists()]
    sample_studies = random.sample(study_ids, min(N_STUDIES_SCAN, len(study_ids)))
    recs, t0 = [], time.time()
    for st in sample_studies:
        for se in train_series.loc[train_series.StudyInstanceUID == st, "SeriesInstanceUID"]:
            files = list_slices(st, se)
            if not files: continue
            ds = pydicom.dcmread(files[0], stop_before_pixels=True, force=True)
            g = lambda k, d=None: getattr(ds, k, d)
            ps = g("PixelSpacing", [np.nan, np.nan])
            recs.append(dict(
                StudyInstanceUID=st, SeriesInstanceUID=se, n_slices=len(files),
                Rows=g("Rows"), Columns=g("Columns"),
                px_y=float(ps[0]) if ps is not None else np.nan,
                px_x=float(ps[1]) if ps is not None else np.nan,
                SliceThickness=float(g("SliceThickness") or np.nan),
                Manufacturer=str(g("Manufacturer", "?")).strip(),
                Model=str(g("ManufacturerModelName", "?")).strip(),
                FieldStrength=g("MagneticFieldStrength"),
                TR=g("RepetitionTime"), TE=g("EchoTime"),
                SeriesDescription=str(g("SeriesDescription", "")).strip(),
                BitsStored=g("BitsStored"), PhotometricInterpretation=g("PhotometricInterpretation"),
                TransferSyntax=str(getattr(ds.file_meta, "TransferSyntaxUID", "?").name
                                   if hasattr(ds, "file_meta") else "?"),
                n_tags=len(ds),
            ))
    meta = pd.DataFrame(recs)
    print(f"scanned {len(sample_studies)} studies / {len(meta)} series in {time.time()-t0:.1f}s")
    display(meta.head())
else:
    meta = pd.DataFrame()

In [ ]:
if len(meta):
    fig, ax = plt.subplots(2, 3, figsize=(13, 6.5))
    ax = ax.ravel()
    ax[0].hist(meta.n_slices, bins=50, color="#4C72B0"); ax[0].set_title("slices per series"); ax[0].set_yscale("log")
    ax[1].hist(meta.Rows.dropna(), bins=40, color="#DD8452"); ax[1].set_title("matrix Rows")
    ax[2].hist(meta.px_x.dropna(), bins=40, color="#55A868"); ax[2].set_title("in-plane pixel spacing (mm)")
    ax[3].hist(meta.SliceThickness.dropna(), bins=40, color="#C44E52"); ax[3].set_title("slice thickness (mm)")
    fs = meta.FieldStrength.dropna().astype(float)
    ax[4].hist(fs, bins=30, color="#8172B3"); ax[4].set_title("field strength (T)")
    mf = meta.Manufacturer.replace("", "?").value_counts().head(8)
    ax[5].barh(mf.index[::-1], mf.values[::-1], color="#937860"); ax[5].set_title("manufacturer")
    plt.tight_layout(); plt.show()

    print(meta[["n_slices", "Rows", "Columns", "px_x", "SliceThickness"]].describe().round(2))

In [ ]:
if len(meta):
    print("TRANSFER SYNTAX MIX (drives decode speed — see benchmark below)")
    display(meta.TransferSyntax.value_counts().to_frame("n_series"))
    print("\nPHOTOMETRIC / BIT DEPTH")
    display(pd.crosstab(meta.PhotometricInterpretation, meta.BitsStored))
    print("\nTOP SERIES DESCRIPTIONS (free text, useful as a weak protocol feature)")
    display(meta.SeriesDescription.str.lower().value_counts().head(25).to_frame("n"))

In [ ]:
# decode-cost benchmark by transfer syntax — plan your preprocessing budget with this
if HAS_DCM and len(meta):
    times = defaultdict(list)
    for _, r in meta.sample(min(40, len(meta)), random_state=SEED).iterrows():
        f = list_slices(r.StudyInstanceUID, r.SeriesInstanceUID)
        if not f: continue
        t0 = time.time()
        try:
            _ = pydicom.dcmread(f[len(f)//2], force=True).pixel_array
            times[r.TransferSyntax].append((time.time() - t0) * 1000)
        except Exception as e:
            times[f"FAILED::{type(e).__name__}"].append(np.nan)
    for k, v in times.items():
        print(f"{k:<45} n={len(v):3d}  median {np.nanmedian(v):7.1f} ms/slice")
    print("\nIf a syntax FAILS, install pylibjpeg / pylibjpeg-openjpeg / gdcm.")

In [ ]:
def load_series(study, series, max_slices=None):
    files = list_slices(study, series)
    if not files: return None
    ds_list = []
    for f in files:
        try:
            d = pydicom.dcmread(f, force=True)
            _ = d.pixel_array
            ds_list.append(d)
        except Exception:
            continue
    if not ds_list: return None
    ds_list.sort(key=lambda d: float(getattr(d, "InstanceNumber", 0) or 0))
    if max_slices and len(ds_list) > max_slices:
        idx = np.linspace(0, len(ds_list) - 1, max_slices).astype(int)
        ds_list = [ds_list[i] for i in idx]
    vol = []
    for d in ds_list:
        a = d.pixel_array.astype(np.float32)
        a = a * float(getattr(d, "RescaleSlope", 1) or 1) + float(getattr(d, "RescaleIntercept", 0) or 0)
        if getattr(d, "PhotometricInterpretation", "") == "MONOCHROME1":
            a = a.max() - a
        vol.append(a)
    shapes = {v.shape for v in vol}
    if len(shapes) > 1:                       # rare, but it happens
        h = min(v.shape[0] for v in vol); w = min(v.shape[1] for v in vol)
        vol = [v[:h, :w] for v in vol]
    return np.stack(vol)

def window(x, lo=1, hi=99):
    a, b = np.percentile(x, [lo, hi])
    return np.clip((x - a) / max(b - a, 1e-6), 0, 1)

def montage(study, series, n=8, title=""):
    v = load_series(study, series, max_slices=n)
    if v is None:
        print("could not load", series); return
    v = window(v)
    fig, ax = plt.subplots(1, len(v), figsize=(2 * len(v), 2.3))
    ax = np.atleast_1d(ax)
    for i, a in enumerate(ax):
        a.imshow(v[i], cmap="gray"); a.axis("off"); a.set_title(f"{i}", fontsize=7)
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()

In [ ]:
# visualise one full study: every series, middle slices
if HAS_DCM and SERIES_DIR.exists():
    demo = sample_studies[0]
    print("STUDY:", demo)
    row = train.loc[train.StudyInstanceUID == demo]
    if len(row) and row[LABELS].notna().all(axis=1).iloc[0]:
        pos = [l for l in LABELS if row.iloc[0][l] == 1]
        print("labels:", pos if pos else "NORMAL")
    for _, s in train_series[train_series.StudyInstanceUID == demo].iterrows():
        montage(demo, s.SeriesInstanceUID, n=6,
                title=f"{s.Anatomical_Plane} | fluid={s.Fluid_Sensitive} fatsat={s.Fat_Suppression}")

In [ ]:
# side-by-side: a positive vs a negative case for one finding
def first_study_with(lab, val=1, plane="Sagittal", fluid=1):
    cand = labeled.loc[labeled[lab] == val, "StudyInstanceUID"]
    for st in cand:
        if not (SERIES_DIR / st).exists(): continue
        ss = train_series[(train_series.StudyInstanceUID == st) &
                          (train_series.Anatomical_Plane == plane) &
                          (train_series.Fluid_Sensitive == fluid)]
        if len(ss): return st, ss.iloc[0].SeriesInstanceUID
    return None, None

if HAS_DCM and SERIES_DIR.exists():
    for lab, plane in [("ACL", "Sagittal"), ("Effusion", "Axial"), ("Medial Meniscus", "Coronal")]:
        for val in (1, 0):
            st, se = first_study_with(lab, val, plane)
            if st:
                montage(st, se, n=6, title=f"{lab} = {val}  ({plane}, fluid-sensitive)")

In [ ]:
# intensity statistics: why per-series normalisation is mandatory
if HAS_DCM and len(meta):
    stats = []
    for _, r in meta.sample(min(25, len(meta)), random_state=SEED).iterrows():
        v = load_series(r.StudyInstanceUID, r.SeriesInstanceUID, max_slices=5)
        if v is None: continue
        stats.append(dict(series=r.SeriesInstanceUID[-8:], mn=v.min(), p1=np.percentile(v, 1),
                          med=np.median(v), p99=np.percentile(v, 99), mx=v.max(),
                          fluid=r.SeriesInstanceUID))
    S = pd.DataFrame(stats)
    display(S[["series", "mn", "p1", "med", "p99", "mx"]].round(1))
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.boxplot([[s] for s in S.p99], vert=False, widths=.5)
    ax.set_xlabel("99th-percentile intensity per series"); ax.set_xscale("log")
    ax.set_title("Raw intensity scale varies by orders of magnitude across series")
    plt.tight_layout(); plt.show()

**Takeaway (pixels).** MRI intensities carry no absolute meaning — the same tissue can be 200 on one scanner and 4000 on another. Always normalise **per series** (percentile clip 0.5–99.5, then z-score or 0–1 scale). Resample to a common in-plane spacing if you crop, and remember slice thickness varies, so 3D kernels see different physical depths across studies.

Decoding is likely your real bottleneck. Convert everything once to compressed `.npy`/HDF5 at fixed resolution before training rather than reading DICOM in the dataloader.

---
## 6. Train vs test: is the test set the same animal?

In [ ]:
print("test studies:", len(test), "| test series rows:", len(test_series))
display(test.head())
display(sample_sub.head(3))
print("\nsubmission columns match spec:",
      list(sample_sub.columns) == ["StudyInstanceUID"] + LABELS)

In [ ]:
cmp = pd.DataFrame({
    "train": train_series.Anatomical_Plane.value_counts(normalize=True),
    "test":  test_series.Anatomical_Plane.value_counts(normalize=True),
}).fillna(0).round(3)
display(cmp)

cmp2 = pd.DataFrame({
    "train": train_series.groupby("StudyInstanceUID").size().describe(),
    "test":  test_series.groupby("StudyInstanceUID").size().describe(),
}).round(2)
display(cmp2)
print("NOTE: the public example test files contain only 3 studies — these comparisons are")
print("indicative only. The real test set is ~1300 studies and is swapped in at scoring time.")

**Takeaway (shift).** The organisers state explicitly that **prevalence is not guaranteed to match across train / public LB / private LB**. Two consequences:

* AUC is prevalence-insensitive in expectation, which protects you — but its *variance* explodes when a label has few positives in the hidden set. Do not chase small LB moves on rare labels.
* Never calibrate thresholds to training prevalence. Submit raw ranking scores; the metric only cares about order.

---
## 7. Summary — what the EDA implies for the solution

**Data structure**
- Prediction unit = **study**; a study holds several series; a series holds 20–45 slices (long tail to hundreds).
- Join key: `StudyInstanceUID` for labels, `StudyInstanceUID` + `SeriesInstanceUID` for the folder path.

**Labels**
- Only a minority of training studies are labeled; the rest ship with a report instead. Multi-label, heavily imbalanced, strongly correlated within clinical groups.
- Metric is macro AUC, so **rare labels count as much as common ones** — and dominate the variance.